In [1]:
# Atividade 10:
# Crie uma função em Python que, com nível de significância de 5%, construa um
# teste de hipótese analisando se o  retorno esperado médio  será superior  ou
# igual à x% (a ser definido pelo usuário) baseado na amostra que você utilizou.
# Realize esta análise para todas as criptomoedas do dataset.

# Atividade 11:
# Realize análises de variância (ANOVA) para comparar os retornos médios diários
# das criptomoedas.
# a. Aplique ANOVA para verificar se o retorno médio diário difere entre as
# criptomoedas analisadas. Caso o resultado seja significativo, realize um teste
# post hoc para identificar quais moedas diferem entre si.
# b. Agrupe as criptomoedas com base em alguma característica comum (ex:
# volatilidade, volume médio negociado, ou retorno médio) e aplique ANOVA
# para verificar se o retorno médio diário difere entre os grupos formados. Caso
# o resultado seja significativo, realize um teste post hoc.

import pandas as pd
import numpy as np
from scipy import stats
import os
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Função para calcular os retornos diários a partir dos preços previsto
def calculate_daily_returns(df, price_column='PREDICTED'):
    """
    Calcula os retornos diários a partir do preço previsto.

    Args:
        df (pd.DataFrame) : DataFrame com os dados da criptomoeda.
        price_column (str): O nome da coluna que contém os preços previstos.

    Returns:
        pd.Series: Uma série contendo os retornos diários. Retorna uma série vazia
                   se a coluna de preço não existir ou for vazia/nula.
    """
    if price_column not in df.columns or df[price_column].isnull().all():
        print(f"Coluna '{price_column}' não encontrada ou contém apenas valores nulos.")
        return pd.Series(dtype=float)

    # Certifica-se de que a coluna de preço é numérica
    df[price_column] = pd.to_numeric(df[price_column], errors='coerce')

    # Remove linhas com valores nulos na coluna de preço após a conversão
    df_clean = df.dropna(subset=[price_column])

    if len(df_clean) < 2:
         print(f"Dados insuficientes para calcular retornos.")
         return pd.Series(dtype=float)

    # Calcula os retornos diários: (Preço Atual - Preço Anterior) / Preço Anterior
    returns = df_clean[price_column].pct_change().dropna()

    return returns

def perform_hypothesis_test(data, hypothesized_annual_return, alpha=0.05):
    """
    Realiza um teste de hipótese unilateral para a média dos retornos diários.

    Args:
        data (pd.Series): Série contendo os dados de retorno diário de uma criptomoeda.
        hypothesized_annual_return (float): O retorno médio anualizado hipotético (em decimal).
        alpha (float): O nível de significância.

    Returns:
        tuple: Uma tupla contendo:
            - float: O valor t calculado.
            - float: O valor p do teste.
            - bool: True se a hipótese nula for rejeitada, False caso contrário.
    """
    # Remove valores NaN da série de dados
    data = data.dropna()

    if len(data) <= 1:
        return np.nan, np.nan, False # Não é possível realizar o teste com 1 ou menos amostras válidas

    # Anualizando o retorno hipotético diário
    # Assumindo 252 dias úteis no ano para retorno anualizado, mas para retorno diário
    # é melhor usar a média diária e comparar com a média diária esperada.
    # A hipótese é sobre a média do retorno diário que levaria ao retorno anualizado x%.
    # Retorno Anualizado = (1 + Retorno Médio Diário)^(Número de dias de negociação no ano) - 1
    # Retorno Médio Diário Esperado = (1 + Retorno Anualizado Hipotético)^(1 / Número de dias de negociação no ano) - 1
    num_trading_days_year = 252 # Um valor comum para ativos financeiros. Ajuste se necessário.
    hypothesized_daily_return = (1 + hypothesized_annual_return)**(1/num_trading_days_year) - 1

    # Estatísticas da amostra
    sample_mean = data.mean()
    sample_std = data.std()
    n = len(data)

    # Cálculo do estatística t
    # H0: mu <= hypothesized_daily_return
    # Ha: mu > hypothesized_daily_return
    if sample_std == 0: # Evitar divisão por zero se todos os retornos forem iguais
        t_statistic = np.inf if sample_mean > hypothesized_daily_return else (-np.inf if sample_mean < hypothesized_daily_return else 0)
    else:
        t_statistic = (sample_mean - hypothesized_daily_return) / (sample_std / np.sqrt(n))

    # Cálculo do valor p para um teste unilateral superior (Ha: mu > mu0)
    # O valor p é a probabilidade de observar um valor t igual ou mais extremo
    # (neste caso, maior) do que o t_statistic calculado, sob a hipótese nula.
    p_value = 1 - stats.t.cdf(t_statistic, df=n-1)

    # Decisão sobre a hipótese nula
    reject_null = p_value < alpha

    return t_statistic, p_value, reject_null

def analyze_all_cryptos_hypothesis(dataframes_dict, hypothesized_annual_return_percent=0.0, alpha=0.05, price_column='PREDICTED'):
    """
    Realiza o teste de hipótese para cada criptomoeda fornecida em um dicionário de DataFrames.

    Args:
        dataframes_dict (dict): Um dicionário onde as chaves são os nomes das criptomoedas
                                e os valores são os DataFrames correspondentes.
        hypothesized_annual_return_percent (float): O retorno médio anualizado hipotético
                                                 em porcentagem (ex: 5 para 5%).
        alpha (float): O nível de significância.
        price_column (str): O nome da coluna que contém os preços de fechamento para calcular retornos.

    Returns:
        pd.DataFrame: Um DataFrame com os resultados do teste de hipótese para cada criptomoeda.
    """
    results = {}
    hypothesized_annual_return = hypothesized_annual_return_percent / 100.0

    for crypto_name, df in dataframes_dict.items():
        print(f"Analisando {crypto_name} usando a coluna '{price_column}'...")
        # Calcular retornos diários para o DataFrame atual
        daily_returns = calculate_daily_returns(df, price_column=price_column)

        if not daily_returns.empty:
            t_stat, p_val, reject_null = perform_hypothesis_test(
                daily_returns,
                hypothesized_annual_return,
                alpha
            )
            results[crypto_name] = {
                'Hypothesized Annual Return (%)': hypothesized_annual_return_percent,
                'Sample Mean Daily Return': daily_returns.mean(),
                'Sample Std Daily Return': daily_returns.std(),
                'Number of Samples': len(daily_returns),
                'T-Statistic': t_stat,
                'P-Value': p_val,
                f'Reject H0 (alpha={alpha*100}%)': reject_null
            }
        else:
             results[crypto_name] = {
                'Hypothesized Annual Return (%)': hypothesized_annual_return_percent,
                'Sample Mean Daily Return': np.nan,
                'Sample Std Daily Return': np.nan,
                'Number of Samples': 0,
                'T-Statistic': np.nan,
                'P-Value': np.nan,
                f'Reject H0 (alpha={alpha*100}%)': False # Não rejeita se não há dados de retorno
            }


    results_df = pd.DataFrame.from_dict(results, orient='index')
    return results_df

# --- Carregar os DataFrames ---
file_paths = {
    'AAVEBTC':  'analises/analise_mlp/AAVEBTC.xlsx',
    'AAVEUSDT': 'analises/analise_mlp/AAVEUSDT.xlsx',
    'ACMUSDD':  'analises/analise_mlp/ACMUSDD.xlsx',
    'ADAUSDT':  'analises/analise_mlp/ADAUSDT.xlsx',
    'BNBUSDT':  'analises/analise_mlp/BNBUSDT.xlsx',
    'BNTUSDT':  'analises/analise_mlp/BNTUSDT.xlsx',
    'CVTBTC':   'analises/analise_mlp/CVTBTC.xlsx',
    'DOGE':     'analises/analise_mlp/DOGE.xlsx',
    'ETCETH':   'analises/analise_mlp/ETCETH.xlsx',
    'USDPUSDT': 'analises/analise_mlp/USDPUSDT.xlsx'
}

dataframes = {}
for name, path in file_paths.items():
    if os.path.exists(path):
        try:
            dataframes[name] = pd.read_excel(path)
            print(f"DataFrame '{name}' carregado com sucesso.")
        except Exception as e:
            print(f"Erro ao carregar '{name}' de '{path}': {e}")
    else:
        print(f"Arquivo '{path}' não encontrado para '{name}'. Pulando...")


# --- Executar a análise ---
# Defina o retorno anualizado hipotético que você quer testar (em porcentagem)
hypothesized_target_annual_return = input('Retorno esperado médio(%): ')
hypothesized_target_annual_return = float(hypothesized_target_annual_return)

# Análise para todas as criptomoedas carregadas, usando a coluna 'PREDICTED'
if dataframes:
    hypothesis_results_predicted = analyze_all_cryptos_hypothesis(
        dataframes,
        hypothesized_annual_return_percent=hypothesized_target_annual_return,
        alpha=0.05,
        price_column='PREDICTED' # Especificado para usar a coluna 'PREDICTED'
    )

    # Exiba os resultados
    print(f"\nResultados do Teste de Hipótese usando a coluna 'PREDICTED'(H0: Retorno Anual Médio <= {hypothesized_target_annual_return}%)")
    display(hypothesis_results_predicted)
else:
    print("Nenhum DataFrame foi carregado. Não é possível realizar a análise.")

# Interpretação dos resultados:
#
# Para cada criptomoeda listada na tabela de resultados:
# - 'Hypothesized Annual Return (%)': O retorno anualizado que está sendo testado.
# - 'Sample Mean Daily Return': O retorno diário médio observado na sua amostra de dados.
# - 'Sample Std Daily Return': O desvio padrão dos retornos diários na sua amostra.
# - 'Number of Samples': O número de retornos diários válidos usados no teste.
# - 'T-Statistic': O valor calculado da estatística t.
# - 'P-Value': A probabilidade de observar um valor t igual ou mais extremo
#              do que o t_statistic calculado, sob a hipótese nula.
# - 'Reject H0': Indica se a hipótese nula é rejeitada com um nível de significância de 5%.
#                Se for True, há evidências estatísticas para acreditar que o retorno anual
#                médio é estatisticamente superior ao "Retorno esperado médio(%) informado.
#                Se for False, não há evidências suficientes na amostra para afirmar isso
#                com 95% de confiança.

DataFrame 'AAVEBTC' carregado com sucesso.
DataFrame 'AAVEUSDT' carregado com sucesso.
DataFrame 'ACMUSDD' carregado com sucesso.
DataFrame 'ADAUSDT' carregado com sucesso.
DataFrame 'BNBUSDT' carregado com sucesso.
DataFrame 'BNTUSDT' carregado com sucesso.
DataFrame 'CVTBTC' carregado com sucesso.
DataFrame 'DOGE' carregado com sucesso.
DataFrame 'ETCETH' carregado com sucesso.
DataFrame 'USDPUSDT' carregado com sucesso.
Analisando AAVEBTC usando a coluna 'PREDICTED'...
Analisando AAVEUSDT usando a coluna 'PREDICTED'...
Analisando ACMUSDD usando a coluna 'PREDICTED'...
Analisando ADAUSDT usando a coluna 'PREDICTED'...
Analisando BNBUSDT usando a coluna 'PREDICTED'...
Analisando BNTUSDT usando a coluna 'PREDICTED'...
Analisando CVTBTC usando a coluna 'PREDICTED'...
Analisando DOGE usando a coluna 'PREDICTED'...
Analisando ETCETH usando a coluna 'PREDICTED'...
Analisando USDPUSDT usando a coluna 'PREDICTED'...

Resultados do Teste de Hipótese usando a coluna 'PREDICTED'(H0: Retorno Anu

,Hypothesized Annual Return (%),Sample Mean Daily Return,Sample Std Daily Return,Number of Samples,T-Statistic,P-Value,Reject H0 (alpha=5.0%)
AAVEBTC,5.0,0.119272,20.202940,41405,1.199343,0.115201,False
AAVEUSDT,5.0,0.000961,0.044129,41459,3.538655,0.000201,True
ACMUSDD,5.0,0.002415,0.164193,13116,1.549605,0.060630,False
ADAUSDT,5.0,0.000043,0.011863,30567,-2.216030,0.986652,False
BNBUSDT,5.0,0.000753,0.036942,44473,3.194422,0.000701,True
BNTUSDT,5.0,-0.000409,0.080332,24231,-1.167186,0.878427,False
CVTBTC,5.0,0.509370,74.789805,41019,1.378854,0.083974,False
DOGE,5.0,-0.535756,92.156177,100296,-1.841795,0.967246,False
ETCETH,5.0,0.002354,0.265113,78354,2.280602,0.011287,True
USDPUSDT,5.0,0.000272,0.025543,27095,0.503005,0.307482,False


In [4]:
# a. Aplicar ANOVA para comparar os retornos médios diários entre as criptomoedas

# Coletar todos os retornos diários de todas as criptomoedas em uma única estrutura
# e adicionar um identificador para cada criptomoeda.
all_returns = []
crypto_labels = []

for name, df in dataframes.items():
    # Assumindo que 'calculate_daily_returns' já foi definida e funciona
    daily_returns = calculate_daily_returns(df, price_column='PREDICTED') # Usando 'PREDICTED' conforme análise anterior
    if not daily_returns.empty:
        all_returns.append(daily_returns.dropna()) # Remove NaNs antes de empilhar
        crypto_labels.extend([name] * len(daily_returns.dropna()))

# Concatenar todos os retornos e labels em um DataFrame para fácil manipulação
if all_returns:
    all_returns_concat = pd.concat(all_returns, ignore_index=True)
    returns_df = pd.DataFrame({'Return': all_returns_concat, 'Crypto': crypto_labels})

    if len(returns_df) > 0 and len(returns_df['Crypto'].unique()) > 1:
        # Realizar o teste ANOVA
        # ANOVA requer que os dados sejam passados como argumentos separados para cada grupo
        # Usando .apply(list) para obter uma lista de arrays, e desempacotando-a com *
        groups = returns_df.groupby('Crypto')['Return'].apply(list)
        anova_result = f_oneway(*groups) # Correção: Desempacotar a lista de arrays

        print("\n--- Análise de Variância (ANOVA) entre Criptomoedas ---")
        print(f"Estatística F: {anova_result.statistic:.4f}")
        print(f"Valor p: {anova_result.pvalue:.4f}")

        # Interpretar o resultado da ANOVA
        alpha_anova = 0.05
        if anova_result.pvalue < alpha_anova:
            print(f"O resultado da ANOVA é estatisticamente significativo (p < {alpha_anova}).")
            print("Há evidências para rejeitar a hipótese nula de que os retornos médios diários são iguais entre todas as criptomoedas.")
            print("\nRealizando teste post hoc (Tukey HSD) para identificar quais pares de moedas diferem...")

            # Realizar o teste post hoc (Tukey HSD)
            # Tukey HSD é apropriado para comparações parciais após ANOVA.
            try:
                tukey_result = pairwise_tukeyhsd(endog=returns_df['Return'], groups=returns_df['Crypto'], alpha=alpha_anova)
                print(tukey_result)

                # Opcional: Visualizar o resultado do Tukey HSD
                tukey_result.plot_simultaneous()
                plt.title("Comparação Múltipla de Retornos Médios (Tukey HSD)")
                plt.show()

            except ValueError as e:
                 print(f"Erro ao realizar o teste Tukey HSD: {e}")
                 print("Verifique se há grupos com apenas uma amostra ou variância zero após a remoção de NaNs.")

        else:
            print(f"O resultado da ANOVA NÃO é estatisticamente significativo (p >= {alpha_anova}).")
            print("Não há evidências suficientes para rejeitar a hipótese nula de que os retornos médios diários são iguais entre todas as criptomoedas.")

    else:
        print("\nDados insuficientes ou apenas uma criptomoeda disponível para realizar a ANOVA entre moedas.")
else:
    print("\nNenhum retorno diário válido encontrado em nenhum DataFrame para realizar a ANOVA entre moedas.")


# b. Agrupar criptomoedas e aplicar ANOVA

# Exemplo de agrupamento: Agrupar por volatilidade (desvio padrão dos retornos diários)
# Calcular a volatilidade (desvio padrão) para cada criptomoeda
volatility = {}
for name, df in dataframes.items():
    daily_returns = calculate_daily_returns(df, price_column='PREDICTED') # Usando 'PREDICTED'
    if not daily_returns.empty:
        volatility[name] = daily_returns.std()
    else:
        volatility[name] = np.nan # Marcar como NaN se não houver retornos

# Criar um DataFrame de volatilidade e remover NaNs
volatility_df = pd.DataFrame(list(volatility.items()), columns=['Crypto', 'Volatility']).dropna()

if len(volatility_df) > 1:
    # Agrupar em 2 grupos: Baixa Volatilidade e Alta Volatilidade (usando a mediana como ponto de corte)
    median_volatility = volatility_df['Volatility'].median()

    volatility_df['Volatility_Group'] = volatility_df['Volatility'].apply(
        lambda x: 'Low Volatility' if x <= median_volatility else 'High Volatility'
    )

    print("\n--- Agrupamento por Volatilidade ---")
    print(volatility_df)

    # Preparar dados para ANOVA baseada nos grupos de volatilidade
    # Precisamos associar cada retorno individual ao seu grupo de volatilidade
    returns_with_group = returns_df.merge(volatility_df[['Crypto', 'Volatility_Group']], on='Crypto', how='left').dropna(subset=['Volatility_Group'])

    if len(returns_with_group) > 0 and len(returns_with_group['Volatility_Group'].unique()) > 1:
        # Realizar ANOVA entre os grupos de volatilidade
        groups_by_volatility = returns_with_group.groupby('Volatility_Group')['Return'].apply(list)
        anova_group_result = f_oneway(*groups_by_volatility) # Correção: Desempacotar a lista de arrays

        print("\n--- Análise de Variância (ANOVA) entre Grupos de Volatilidade ---")
        print(f"Estatística F: {anova_group_result.statistic:.4f}")
        print(f"Valor p: {anova_group_result.pvalue:.4f}")

        # Interpretar o resultado da ANOVA entre grupos
        alpha_anova_group = 0.05
        if anova_group_result.pvalue < alpha_anova_group:
            print(f"O resultado da ANOVA entre grupos é estatisticamente significativo (p < {alpha_anova_group}).")
            print("Há evidências para rejeitar a hipótese nula de que os retornos médios diários são iguais entre os grupos de volatilidade.")
            print("\nRealizando teste post hoc (Tukey HSD) para identificar quais grupos diferem...")

            # Realizar o teste post hoc (Tukey HSD) para os grupos
            try:
                 tukey_group_result = pairwise_tukeyhsd(endog=returns_with_group['Return'], groups=returns_with_group['Volatility_Group'], alpha=alpha_anova_group)
                 print(tukey_group_result)

                 # Opcional: Visualizar o resultado do Tukey HSD para grupos
                 tukey_group_result.plot_simultaneous()
                 plt.title("Comparação Múltipla de Retornos Médios por Grupo de Volatilidade (Tukey HSD)")
                 plt.show()

            except ValueError as e:
                 print(f"Erro ao realizar o teste Tukey HSD para grupos: {e}")
                 print("Verifique se há grupos com apenas uma amostra ou variância zero após a remoção de NaNs.")

        else:
            print(f"O resultado da ANOVA entre grupos NÃO é estatisticamente significativo (p >= {alpha_anova_group}).")
            print("Não há evidências suficientes para rejeitar a hipótese nula de que os retornos médios diários são iguais entre os grupos de volatilidade.")

    else:
         print("\nDados insuficientes ou apenas um grupo de volatilidade disponível para realizar a ANOVA.")

else:
    print("\nDados de volatilidade insuficientes (menos de 2 criptomoedas com retornos válidos) para realizar agrupamento e ANOVA.")


--- Análise de Variância (ANOVA) entre Criptomoedas ---
Estatística F: 1.7127
Valor p: 0.0802
O resultado da ANOVA NÃO é estatisticamente significativo (p >= 0.05).
Não há evidências suficientes para rejeitar a hipótese nula de que os retornos médios diários são iguais entre todas as criptomoedas.

--- Agrupamento por Volatilidade ---
     Crypto  Volatility Volatility_Group
0   AAVEBTC   20.202940  High Volatility
1  AAVEUSDT    0.044129   Low Volatility
2   ACMUSDD    0.164193  High Volatility
3   ADAUSDT    0.011863   Low Volatility
4   BNBUSDT    0.036942   Low Volatility
5   BNTUSDT    0.080332   Low Volatility
6    CVTBTC   74.789805  High Volatility
7      DOGE   92.156177  High Volatility
8    ETCETH    0.265113  High Volatility
9  USDPUSDT    0.025543   Low Volatility

--- Análise de Variância (ANOVA) entre Grupos de Volatilidade ---
Estatística F: 0.4309
Valor p: 0.5116
O resultado da ANOVA entre grupos NÃO é estatisticamente significativo (p >= 0.05).
Não há evidências sufi